# 单因子测试框架

## 策略配置
- **股票池** (`cn_stock_prefactors_community`)
  - 按 `total_market_cap` 升序取前500只
  - 排除 ST、停牌、北交所
- **持仓**
  - 指标=1 时持有，等权分配，每日调仓

## 指标定义
### 预期ST
**预期ST = 前年年报亏损 AND 去年年报预亏公告**
- **条件1:前年年报亏损** (`cn_stock_factors_financial_items`)
  - 取去年最后一个交易日的 `net_profit_ly` < 0
- **条件2:收到去年年报预亏公告** (`cn_stock_profit_estimate`）
  - MONTH(end_date) = 12              -- 年报预告的end_date在12月
  - AND fore_profit < 0               -- 预亏
  - AND YEAR(end_date) = YEAR(date) - 1  -- 预告去年年报
  - 按 `(instrument, YEAR(date))` 分区，LAST IGNORE NULLS 填充
- **条件3:年报尚未披露** (`cn_stock_factors_financial_items`)
  - 年报披露日期 = `net_profit_ly` 字段值发生变化的首个日期（1-4月内，每年取第一次）
  - date < COALESCE(annual_report_date, 当年5月1日)
  - 若披露日期缺失（部分股票 net_profit_ly 无变化），默认 4 月 30 日截止

## 无未来数据保证
- `net_profit_ly`: PIT数据，当日可知的最新年报利润
- `annual_report_publish`: 预计算的年报披露日期（避免incremental模式问题）
- `profit_estimate.date`: 公告日期，无未来信息
- **注意**: WINDOW函数`LAST IGNORE NULLS`在incremental模式下需测试验证


In [7]:
from bigmodule import M, I
import dai
import pandas as pd


def m5_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))


def m5_before_trading_start_bigquant_run(context, data):
    pass


def m5_handle_tick_bigquant_run(context, tick):
    pass


def m5_handle_data_bigquant_run(context, data):
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]
    target_instruments = set(today_df["instrument"])
    holding_instruments = set(context.get_account_positions().keys())

    for instrument in holding_instruments - target_instruments:
        context.order_target_percent(instrument, 0)

    for i, x in today_df.iterrows():
        position = 0.0 if pd.isnull(x.position) else float(x.position)
        context.order_target_percent(x.instrument, position)


def m5_handle_trade_bigquant_run(context, trade):
    pass


def m5_handle_order_bigquant_run(context, order):
    pass


def m5_after_trading_bigquant_run(context, data):
    pass


# ========== 数据准备 ==========
# 预期ST因子:去年年报亏损 AND 当年预亏公告（年报披露前有效）

# ==================== 预计算数据（只需运行一次）====================
# 预计算年报披露日期和 net_profit_ly，避免在策略 SQL 中 JOIN 大表

# 1. 年报披露日期（通过 net_profit_ly 变化推断，每年取第一次变化日期）
ANNUAL_REPORT_SQL = """
SELECT DISTINCT ON (instrument, YEAR(date))
    date AS publish_date,
    instrument,
    YEAR(date) AS publish_year
FROM (
    SELECT 
        date,
        instrument,
        net_profit_ly,
        LAG(net_profit_ly) OVER (PARTITION BY instrument ORDER BY date) AS prev_profit
    FROM cn_stock_factors_financial_items
)
WHERE net_profit_ly IS DISTINCT FROM prev_profit
  AND net_profit_ly IS NOT NULL
  AND MONTH(date) IN (1,2,3,4)
ORDER BY instrument, YEAR(date), date
"""

print("预计算年报披露日期...")
annual_report_df = dai.query(ANNUAL_REPORT_SQL, filters={"date": ["2015-01-01", "2030-01-01"]}).df()
print(f"年报披露记录数:{len(annual_report_df)}")

# 2. 预计算 net_profit_ly（只提取需要的字段，大幅减少内存）
NET_PROFIT_SQL = """
SELECT date, instrument, net_profit_ly
FROM cn_stock_factors_financial_items
WHERE net_profit_ly IS NOT NULL
"""

print("预计算 net_profit_ly...")
net_profit_df = dai.query(NET_PROFIT_SQL, filters={"date": ["2015-01-01", "2030-01-01"]}).df()
print(f"net_profit_ly 记录数:{len(net_profit_df)}")

# ==================== 指标定义 ====================

# 预期ST指标SQL
# - annual_report_publish: 预计算的年报披露日期（通过 bind_relations 引用）
# - net_profit_ly: PIT数据，当日可知的最新年报利润，无未来信息
# - profit_estimate: 业绩预告公告日期，无未来信息
INDICATOR_EXPECTED_ST = """
-- 年报披露日期（预计算数据）
annual_report_publish AS (
    SELECT publish_date, instrument, publish_year
    FROM annual_report_ref
),

-- net_profit_ly（预计算数据）
net_profit_data AS (
    SELECT date, instrument, net_profit_ly
    FROM net_profit_ref
),

-- 业绩预告（只取年报预亏，end_date 月份=12，fore_profit < 0）
profit_estimate AS (
    SELECT
        date AS announce_date,
        CASE WHEN CAST(instrument AS INT) >= 600000
             THEN LPAD(CAST(CAST(instrument AS INT) AS VARCHAR), 6, '0') || '.SH'
             ELSE LPAD(CAST(CAST(instrument AS INT) AS VARCHAR), 6, '0') || '.SZ'
        END AS instrument,
        YEAR(end_date) AS fore_year
    FROM cn_stock_profit_estimate
    WHERE MONTH(end_date) = 12 AND fore_profit < 0
),

-- 计算预期ST指标
-- 核心逻辑:使用当日可知的 net_profit_ly（PIT数据，无未来信息）
expected_st AS (
    SELECT
        a.date,
        a.instrument,
        -- 当日可知的最新年报利润（PIT数据，在年报披露前是前年的值）
        b.net_profit_ly AS net_profit_ly_prev,
        -- 预告年份应为去年
        YEAR(a.date) - 1 AS target_fore_year,
        -- LAST 按 (instrument, year) 分区，扫描当年发布的预告
        LAST(d.fore_year IGNORE NULLS) OVER (
            PARTITION BY a.instrument, YEAR(a.date)
            ORDER BY a.date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS latest_fore_year,
        -- 当年年报披露日期
        e.publish_date AS annual_report_date
    FROM cn_stock_prefactors_community a
    LEFT JOIN net_profit_data b USING (date, instrument)
    LEFT JOIN profit_estimate d ON a.instrument = d.instrument AND a.date = d.announce_date
    LEFT JOIN annual_report_publish e ON a.instrument = e.instrument AND YEAR(a.date) = e.publish_year
),

-- 预期ST = 前年年报亏损 AND 当年预亏公告（年报披露前有效）
with_expected_st AS (
    SELECT
        date,
        instrument,
        net_profit_ly_prev,
        target_fore_year,
        latest_fore_year,
        annual_report_date,
        CASE WHEN
            -- 条件1:前年年报亏损（net_profit_ly 是当日可知的最新年报）
            net_profit_ly_prev < 0
            -- 条件2:当年预亏公告
            AND latest_fore_year = target_fore_year
            -- 条件3:当年年报尚未披露（缺失则用4月30日作为默认截止）
            AND date < COALESCE(annual_report_date, MAKE_DATE(YEAR(date), 5, 1))
        THEN 1 ELSE 0 END AS is_expected_st
    FROM expected_st
),
"""

# ==================== 策略SQL ====================

stock_sql = f"""
WITH 
-- 股票池:市值最小500只（非ST、非停牌、非北交所）
small_cap_500 AS (
    SELECT date, instrument, total_market_cap
    FROM cn_stock_prefactors_community
    WHERE st_status = 0 AND suspended = 0 AND is_bz50 = 0
    QUALIFY ROW_NUMBER() OVER (PARTITION BY date ORDER BY total_market_cap ASC) <= 500
),

{INDICATOR_EXPECTED_ST}

-- 筛选:股票池 + 指标=1
filtered AS (
    SELECT 
        a.date,
        a.instrument,
        a.total_market_cap,
        b.net_profit_ly_prev,
        b.latest_fore_year,
        b.annual_report_date,
        b.is_expected_st
    FROM small_cap_500 a
    JOIN with_expected_st b USING (date, instrument)
    WHERE b.is_expected_st = 1
)

SELECT
    date,
    instrument,
    total_market_cap,
    net_profit_ly_prev,
    latest_fore_year,
    annual_report_date,
    is_expected_st,
    -total_market_cap AS score,
    1.0 / c_sum(1) AS position
FROM filtered
ORDER BY date, instrument
"""

print("正在查询数据...")
filtered_df = dai.query(
    stock_sql, 
    filters={"date": ["2020-01-01", "2026-12-31"]},
    bind_relations={
        "annual_report_ref": annual_report_df,
        "net_profit_ref": net_profit_df
    }
).df()
print(f"满足预期ST条件的记录数:{len(filtered_df)}")
print(f"每日平均持仓数量:{filtered_df.groupby('date')['instrument'].count().mean():.1f}")

stock_data_ds = dai.DataSource.write_bdb(filtered_df)

# ========== 回测 ==========
start_date = '2021-01-01'
end_date = '2026-04-07'

m5 = M.bigtrader.v30(
    data=stock_data_ds,
    start_date=start_date,
    end_date=end_date,
    initialize=m5_initialize_bigquant_run,
    before_trading_start=m5_before_trading_start_bigquant_run,
    handle_tick=m5_handle_tick_bigquant_run,
    handle_data=m5_handle_data_bigquant_run,
    handle_trade=m5_handle_trade_bigquant_run,
    handle_order=m5_handle_order_bigquant_run,
    after_trading=m5_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="daily",
    product_type="股票",
    rebalance_period_type="交易日",
    rebalance_period_days="1",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="标准模式",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="open",
    order_price_field_sell="open",
    benchmark="沪深300指数",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="m5"
)

正在查询数据...
满足预期ST条件的记录数:21395
每日平均持仓数量:53.1
[2026-04-09 10:45:36] [info     ] bigtrader.v30 开始运行 ..
[2026-04-09 10:45:36] [info     ] read input 'data' ..
[2026-04-09 10:45:36] [info     ] 2021-01-01, 2026-04-07, , equity, instruments=363
[2026-04-09 10:45:37] [info     ] bigtrader module V2.1.0
[2026-04-09 10:45:37] [info     ] bigtrader engine v0.1.0.post9+g6d7300d 2026-02-10
[2026-04-09 10:45:50] [info     ] backtest done, raw_perf_ds:dai.DataSource("_039ac3427cfb46d4a1ed5cef4626f215")


[2026-04-09 10:45:54] [info     ] bigtrader.v30 运行完成 [17.921s].


In [8]:
# ========== 导出交易记录CSV ==========
# 只记录指标0->1->0的完整hold周期（基于原始指标，不受股票池筛选影响）

raw_perf = m5.raw_perf.read()
trades_list = [t for txns in raw_perf['transactions'] if txns for t in txns]
trades_df = pd.DataFrame(trades_list)

# 查询原始is_expected_st指标（不经过股票池筛选）
traded_instruments = trades_df['symbol'].unique().tolist()
INDICATOR_QUERY = f"""
WITH
{INDICATOR_EXPECTED_ST.rstrip().rstrip(',')}

SELECT date, instrument, is_expected_st
FROM with_expected_st
"""
indicator_df = dai.query(
    INDICATOR_QUERY,
    filters={"date": ["2020-01-01", "2030-01-01"], "instrument": traded_instruments},
    bind_relations={"annual_report_ref": annual_report_df, "net_profit_ref": net_profit_df}
).df()

# 构建指标=1的日期/股票集合（基于原始指标，不受股票池影响）
signal_df = indicator_df[indicator_df['is_expected_st'] == 1]
signal_set = set(zip(signal_df['date'].astype(str), signal_df['instrument']))

# 查询股票名称和行业（取最新数据）
info_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    name
FROM cn_stock_prefactors_community
ORDER BY instrument, date DESC
"""
stock_info = dai.query(info_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
stock_info_dict = stock_info.set_index('instrument')['name'].to_dict()

industry_sql = """
SELECT DISTINCT ON (instrument)
    instrument,
    industry_level1_name,
    industry_level2_name
FROM cn_stock_industry_component
ORDER BY instrument, date DESC
"""
industry_info = dai.query(industry_sql, filters={"date": ["2020-01-01", "2030-01-01"]}).df()
industry_dict = industry_info.set_index('instrument')[['industry_level1_name', 'industry_level2_name']].to_dict('index')

output_records = []
holdings = {}  # instrument -> {'buy_date': str, 'buy_price': float}

for idx, row in trades_df.iterrows():
    instrument = row['symbol']
    dt = pd.to_datetime(row['dt']).strftime('%Y-%m-%d')
    amount = row['amount']
    price = row['price']
    
    in_signal = (dt, instrument) in signal_set
    
    if amount > 0 and instrument not in holdings:
        # 首次买入（指标0->1）
        holdings[instrument] = {'buy_date': dt, 'buy_price': price}
    elif amount < 0 and instrument in holdings and not in_signal:
        # 清仓卖出（指标1->0）:卖出日当天指标已经=0
        buy_info = holdings.pop(instrument)
        pnl = (price - buy_info['buy_price']) / buy_info['buy_price']
        ind = industry_dict.get(instrument, {})
        output_records.append({
            '股票代码': instrument.split('.')[0],
            '股票名': stock_info_dict.get(instrument, ''),
            '行业分类': ind.get('industry_level1_name', ''),
            '二级行业': ind.get('industry_level2_name', ''),
            '买入日期': buy_info['buy_date'],
            '卖出日期': dt,
            '买入价格(前复权)': round(buy_info['buy_price'], 2),
            '卖出价格(前复权)': round(price, 2),
            '涨幅': round(pnl, 4)
        })

output_df = pd.DataFrame(output_records)
output_df = output_df.sort_values('卖出日期', ascending=False)

output_path = './预期ST_bigquant交易记录.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"交易记录已保存到:{output_path}")
print(f"共 {len(output_df)} 条交易记录")
output_df.head(20)

交易记录已保存到:./预期ST_bigquant交易记录.csv
共 1895 条交易记录


,股票代码,股票名,行业分类,二级行业,买入日期,卖出日期,买入价格(前复权),卖出价格(前复权),涨幅
1894,300786,国林科技,机械,专用机械,2026-04-01,2026-04-07,14.31,13.99,-0.0224
1886,300103,达刚控股,机械,专用机械,2026-03-19,2026-04-03,7.66,6.96,-0.0914
1879,300736,百邦科技,通信,通信设备,2026-04-02,2026-04-03,20.00,21.65,0.0825
1880,000663,永安林业,轻工制造,家居,2026-03-19,2026-04-03,7.13,6.75,-0.0533
1881,000702,正虹科技,农林牧渔,畜牧业,2026-03-19,2026-04-03,7.00,6.55,-0.0643
1883,002227,奥 特 迅,电力设备及新能源,新能源动力系统,2026-03-19,2026-04-03,9.69,9.20,-0.0506
1884,002513,蓝丰生化,基础化工,农用化工,2026-03-26,2026-04-03,6.33,6.09,-0.0379
1885,300074,华平股份,计算机,计算机设备,2026-03-19,2026-04-03,4.68,4.18,-0.1068
1882,002105,信隆健康,机械,运输设备,2026-03-19,2026-04-03,6.96,6.34,-0.0891
1887,300126,锐奇股份,机械,通用设备,2026-03-24,2026-04-03,7.60,7.70,0.0132
